In [ ]:
# Locate the Quant checkout before importing its shared configuration.
import os
import platform
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_QUANT_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents) if (candidate / "project_paths.py").is_file()),
    Path(os.environ.get(
        "QUANT_ROOT",
        r"C:\Users\user\Documents\GitHub\Quant"
        if platform.system() == "Windows"
        else "/Users/xinc/GitHub/Quant",
    )),
)
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
from project_paths import DATA_ROOT, NOTE_REPO_ROOT, QUANT_ROOT

# Quant data policy: this repo reads Google Drive data; downloads run from note.
import sys
from pathlib import Path
_QUANT_ROOT = QUANT_ROOT
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
import cloud_data


In [ ]:
# Locate the Quant checkout before importing its shared configuration.
import os
import platform
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_QUANT_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents) if (candidate / "project_paths.py").is_file()),
    Path(os.environ.get(
        "QUANT_ROOT",
        r"C:\Users\user\Documents\GitHub\Quant"
        if platform.system() == "Windows"
        else "/Users/xinc/GitHub/Quant",
    )),
)
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
from project_paths import DATA_ROOT, NOTE_REPO_ROOT, QUANT_ROOT

import gc
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from loguru import logger

REPO_ROOT = QUANT_ROOT
NOTE_ROOT = NOTE_REPO_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(NOTE_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTE_ROOT))
sys.path.append(os.getcwd())

%load_ext autoreload
%autoreload 2

from cloud_data import (
    TW_STOCK_DAILY_PRICE,
    TW_STOCK_DISPOSAL,
    TW_STOCK_DISPOSAL_INFORMATION,
    TW_STOCK_PROCESSED_DISPOSAL,
)
from utils import load_price_ranges, run_event_study, process_disposal_events
from analyzer import DisposalAnalyzer

OFFSET_DAYS = 5
START_DATE = "2018-01-01"
DATA_DIR = TW_STOCK_DISPOSAL
DATA_DIR.mkdir(parents=True, exist_ok=True)


# Setup

## Data Preparation

### 抓取處置股名單 (Finlab)

In [ ]:
finlab_disposal = pd.read_pickle(TW_STOCK_DISPOSAL_INFORMATION)
finlab_disposal["date"] = pd.to_datetime(finlab_disposal["date"])
finlab_disposal = finlab_disposal.loc[finlab_disposal["date"] >= START_DATE]

processed_disposal = process_disposal_events(finlab_disposal)
processed_disposal.to_csv(TW_STOCK_PROCESSED_DISPOSAL, index=False, encoding="utf-8-sig")
print(f"Loaded {len(finlab_disposal):,} local disposal records.")


### 抓取個股股價 (FinMind) - 用 Postgres 就好

In [ ]:
if "processed_disposal" not in locals():
    processed_disposal = pd.read_csv(TW_STOCK_PROCESSED_DISPOSAL)

price_df = load_price_ranges(
    TW_STOCK_DAILY_PRICE,
    processed_disposal,
    offset_days=OFFSET_DAYS,
    max_workers=10,
)
if not price_df.empty:
    save_path = DATA_DIR / "price_df.csv"
    price_df.to_csv(save_path, index=False)
    print(f"Selected {len(price_df):,} local price rows.")
else:
    print("[Warning] No matching local price data.")

gc.collect()


### 抓個股股價(Postgres)

In [ ]:
processed_disposal = pd.read_csv(
    TW_STOCK_PROCESSED_DISPOSAL,
    dtype={"Stock_id": str},
)
price_df = load_price_ranges(
    TW_STOCK_DAILY_PRICE,
    processed_disposal,
    offset_days=OFFSET_DAYS,
)
price_df.to_csv(DATA_DIR / "price_df.csv", index=False)


## 抓大盤 & 指數

In [ ]:
price_df = pd.read_csv(
    DATA_DIR / "price_df.csv",
    dtype={"Stock_id": str},
)
print("Index enrichment must also be prepared by note before it is used here.")


## Event Integration

In [ ]:
if 'price_df' not in locals():
    price_df = pd.read_csv(f'{DATA_DIR}/price_df.csv', dtype={'Stock_id': str})

if 'processed_disposal' not in locals():
    processed_disposal = pd.read_csv(f'{DATA_DIR}/processed_disposal_events.csv', dtype={'Stock_id': str})

# 回傳：
# 1. disposal_wide: 寬表格 (Signal Use)
# 2. disposal_long: 長表格 (Analysis Use)
disposal_wide, disposal_long = run_event_study(price_df, processed_disposal, offset_days=OFFSET_DAYS)

if not disposal_long.empty:
    disposal_wide.to_csv(f'{DATA_DIR}/disposal_df_wide.csv', index=False, encoding='utf-8-sig')
    disposal_long.to_csv(f'{DATA_DIR}/disposal_df_long.csv', index=False, encoding='utf-8-sig')
    
else:
    print("[Error] Event study returned empty result.")

## 僅保留股票資訊

In [ ]:
disposal_long = pd.read_csv(f'{DATA_DIR}/disposal_df_long.csv', dtype={'Stock_id': str}, low_memory=False)

# 篩選條件：優先使用 industry 排除 ETF 與 DR (Cell 15 Modified)
if 'industry' in disposal_long.columns:
    print("Filtering by industry...")
    # 定義要排除的產業
    exclude_industries = [
        'ETF', '存託憑證', '受益證券', 'ETN', '創新板股票', '上櫃指數股票型基金(ETF)'
    ]
    mask_exclude = disposal_long['industry'].isin(exclude_industries)
    
    # 同時過濾掉 91 (DR) 與 00 (ETF) 以防萬一
    s_id = disposal_long['Stock_id'].astype(str)
    mask_id_exclude = (s_id.str.startswith('00')) | (s_id.str.startswith('91'))
    
    disposal_long = disposal_long[~mask_exclude & ~mask_id_exclude]
    print(f"Remaining rows after industry filter: {len(disposal_long)}")
    
else:
    print("[Warning] 'industry' column not found. Fallback to Stock_id filtering.")
    s_id = disposal_long['Stock_id'].astype(str)
    disposal_long = disposal_long[
        (s_id.str.len() == 4) &
        (~s_id.str.startswith('00')) &
        (~s_id.str.startswith('91'))
    ]

if 'Unnamed: 0' in disposal_long.columns:
    disposal_long.drop(columns=['Unnamed: 0'], inplace=True)

disposal_long.dropna(axis=1, how='all').to_csv(f'{DATA_DIR}/disposal_df_long_stock.csv', index=False)

# Analysis

In [ ]:
disposal_long = pd.read_csv(f'{DATA_DIR}/disposal_df_long_stock.csv', parse_dates=['Date'], low_memory=False)
analyzer = DisposalAnalyzer(disposal_long)
# analyzer.display_dataframe()

## overall
`[Disposal Level Statistics]` 中的統計數據基於 **日當沖報酬率** 計算

In [ ]:
analyzer.overall_analysis()

## seperate by trend

In [ ]:
seperated_df = analyzer.seprate_by_trend()

### 日盤

In [ ]:
analyzer.plot_trend_return(df=seperated_df, session='position')

### 夜盤

In [ ]:
analyzer.plot_trend_return(seperated_df, session='after_market')

### 夜 + 日

In [ ]:
analyzer.plot_trend_return(seperated_df, 'all')

## Industry Analysis

In [ ]:
analyzer.plot_3d_return_surface(seperated_df, session='position', bins=20, split_by_direction=True, use_browser=False, show_metrics='mean')

In [ ]:
# given dimension
# slice_by: ["ind_ret", "time"]
analyzer.plot_2d_slice(seperated_df, session='position', slice_by='time', target='s+10')